<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/RNNAudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import os
from google.colab import userdata
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,Bidirectional,LSTM,Dropout
from tensorflow.keras import Sequential
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

In [13]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [14]:
!kaggle datasets download -d uldisvalainis/audio-emotions

Dataset URL: https://www.kaggle.com/datasets/uldisvalainis/audio-emotions
License(s): unknown
audio-emotions.zip: Skipping, found more recently modified local copy (use --force to force download)


In [15]:
!unzip -q audio-emotions.zip -d ./dataset_folder

replace ./dataset_folder/Emotions/Angry/03-01-05-01-01-01-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [16]:
data = []
labels = []
emotions = ['Angry', 'Disgusted', 'Fearful', 'Happy', 'Neutral', 'Sad', 'Suprised']

In [17]:
#Loop through each emotion folder:
for i,emotion in enumerate(emotions):
  folder_path = f'/content/dataset_folder/Emotions/{emotion}'
  for file in os.listdir(folder_path):
    if file.endswith('.wav'):
      file_path = os.path.join(folder_path,file)

      #Load audio(limit 3 seconds)
      audio, sr = librosa.load(file_path,duration=3)

      #Extract MFCCs (40 features)
      mfccs = librosa.feature.mfcc(y=audio,sr=sr,n_mfcc=40)

      # 3. Fix length (Padding/Truncating to 130 time steps)
      if mfccs.shape[1] < 130:
        mfccs = np.pad(mfccs, ((0,0), (0, 130 - mfccs.shape[1])), mode='constant')
      else:
        mfccs = mfccs[:,:130] # here mfccs has shape like (features,timesteps), so we tell that feature should be as it is, but for time steps start from beginning till 130

      data.append(mfccs.T) # now it willbe (timesteps,features)
      labels.append(i)

x = np.array(data)
y = np.array(labels)

In [19]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42,stratify=y)

In [26]:
model = Sequential([
    Input(shape=(130,40)),

    Bidirectional(LSTM(128,dropout=0.2,return_sequences=True)),
    Bidirectional(LSTM(128,dropout=0.2,return_sequences=False)),

    Dense(128,activation='relu'),
    Dense(7,activation='softmax')
])

In [29]:
callback = EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True,verbose=1)

In [30]:
x_train.shape

(8958, 130, 40)

In [31]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(x_train,y_train,validation_data=(x_test,y_test),verbose=1,epochs=10,callbacks=[callback])

Epoch 1/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 245s 842ms/step - accuracy: 0.3523 - loss: 1.5889 - val_accuracy: 0.4677 - val_loss: 1.3024
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 258s 829ms/step - accuracy: 0.4626 - loss: 1.3248 - val_accuracy: 0.5018 - val_loss: 1.2559
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 233s 834ms/step - accuracy: 0.4988 - loss: 1.2357 - val_accuracy: 0.5297 - val_loss: 1.1509
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 279s 893ms/step - accuracy: 0.5108 - loss: 1.2037 - val_accuracy: 0.5453 - val_loss: 1.1257
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 243s 826ms/step - accuracy: 0.5212 - loss: 1.1715 - val_accuracy: 0.5315 - val_loss: 1.2036
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 263s 829ms/step - accuracy: 0.5365 - loss: 1.1479 - val_accuracy: 0.5451 - val_loss: 1.1591
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 252s 902ms/step - accuracy: 0.5416 - loss: 1.1364 - val_accuracy: 0.5693 - val_loss: 1.0708
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 244s 836ms/step - accuracy: 0.5449 -